[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/en_clase/S09_02_pca_en_clase.ipynb)

# S09 · 02 — Reducir dimensiones con PCA · versión en clase

⏱ 30 min en clase · Sprint 9 · M3 (UD 3.10, PCA con mención a t-SNE)

**Contexto TurisData.** Dirección quiere *ver* a sus huéspedes en una sola imagen, pero tenemos 7 variables por persona y no se puede dibujar en 7 dimensiones. **PCA** (análisis de componentes principales) resume muchas variables en pocas "variables nuevas" (componentes) que conservan la mayor parte de la información. Sirve para **visualizar**, para **quitar ruido** y para **acelerar** otros modelos.

**Este cuaderno tiene 2 ejercicios** (1 a 2), los que se hacen en el aula. Las cargas, PCA como paso previo al clustering, la mención a t-SNE, el mini-reto y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1 y 2 son los 2 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

huespedes = pd.read_csv(dato("huespedes.csv"))
variables = [c for c in huespedes.columns if c != "id_huesped"]
X = StandardScaler().fit_transform(huespedes[variables])
grupo = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit_predict(X)
print(X.shape)
print("Listo ✔")

### 👀 A2 · EJEMPLO — La varianza explicada de cada componente

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué porcentaje de la información explica el componente 1 y cuál el 2?
2. ¿Qué componentes explican menos del 5 % cada uno?
3. ¿Por qué los porcentajes están ordenados de mayor a menor?

In [ ]:
pca_completo = PCA(random_state=SEMILLA).fit(X)
varianza = pca_completo.explained_variance_ratio_
for i, v in enumerate(varianza, start=1):
    print(f"Componente {i}: {v:.1%}")

### ✏️ A3 · EJERCICIO 1 de 2 — La varianza acumulada y el 90 %

**Tu turno:** Calcula en `varianza_acum` la varianza explicada **acumulada** (usa `np.cumsum` sobre `varianza`) y en `n_comp_90` el **número mínimo de componentes** que explican al menos el 90 % de la varianza.

**Debe salir:** `con 5 componentes conservamos el 90 % de la información` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Qué valen `n_comp_90` y la varianza acumulada con 4 componentes?
2. ¿Por qué con 4 componentes no basta, si 89,8 % casi es 90 %?
3. ¿Qué ahorro supone quedarse con 5 componentes?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: np.cumsum da la acumulada; para el mínimo de componentes, busca la primera posición que llega a 0,90 y recuerda que se cuenta desde 1
varianza_acum = ...
n_comp_90 = ...

assert abs(varianza_acum[-1] - 1) < 1e-9 and len(varianza_acum) == 7, "varianza_acum debe acabar en 1 y tener 7 valores"
assert n_comp_90 == 5, "Revisa: busca el primer componente donde la acumulada llega a 0.90 y cuéntalo desde 1"
print("con", n_comp_90, "componentes conservamos el 90 % de la información")
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Gráficos de la varianza: por componente y acumulada

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿En qué componente cruza la curva la línea roja del 90 %?
2. ¿Qué ve el gráfico de barras que no ve el de la curva?
3. ¿Qué quiere decir que la curva «se aplane» al final?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(range(1, 8), varianza)
ax[0].set_title("Varianza explicada por componente")
ax[0].set_xlabel("Componente"); ax[0].set_ylabel("Proporción")
ax[1].plot(range(1, 8), varianza_acum, marker="o")
ax[1].axhline(0.9, color="tab:red", linestyle="--", label="90 %")
ax[1].set_title("Varianza explicada acumulada")
ax[1].set_xlabel("Número de componentes"); ax[1].legend()
plt.tight_layout(); plt.show()

### 👀 A5 · EJEMPLO — Proyectar los huéspedes a 2 componentes

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Qué forma tiene `Z2` y qué significa cada dimensión?
2. ¿Cuánta información se conserva con 2 componentes?
3. ¿Qué se pierde al quedarse con 2 componentes?

In [ ]:
pca2 = PCA(n_components=2, random_state=SEMILLA)
Z2 = pca2.fit_transform(X)
print("Forma:", Z2.shape, "· información conservada:", f"{pca2.explained_variance_ratio_.sum():.1%}")

### ✏️ A6 · EJERCICIO 2 de 2 — El mapa de huéspedes en 2D

**Tu turno:** Dibuja un diagrama de dispersión de `Z2` (columna 0 en el eje X, columna 1 en el eje Y) coloreando cada punto según la variable `grupo` (el segmento de k-means). Guarda la figura en `fig2d`. Pon título y nombres de ejes con el porcentaje de varianza de cada componente.

**Debe salir:** un gráfico con 4 nubes de puntos de colores (una por segmento) y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Se ven los 4 segmentos separados? ¿Cuáles son los más cercanos entre sí?
2. ¿Qué indica sobre el clustering que los segmentos salgan separados en el mapa?
3. ¿Por qué conviene poner el % de varianza en los ejes?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
# pista: ax.scatter(Z2[:, 0], Z2[:, 1], c=grupo, cmap="tab10") dibuja los puntos; después ax.set_title, ax.set_xlabel y ax.set_ylabel
fig2d, ax = plt.subplots(figsize=(7, 5))
...

assert len(fig2d.axes[0].collections) >= 1, "fig2d debe contener un scatter"
assert fig2d.axes[0].get_title() != "", "Añade un título al gráfico"
plt.show()
print("✔ Ejercicio 2 correcto")

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_02_pca.ipynb)** trae las cargas de los componentes, PCA como paso previo al clustering, t-SNE, el mini-reto y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn`, el escalado, `PCA`, `KMeans` y la silueta.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Lee `huespedes.csv` (1 200 huéspedes), guarda en `variables` las 7 variables de comportamiento (sin `id_huesped`) y las estandariza en la matriz `X` (PCA se hace **siempre** con variables escaladas).
    - Agrupa los huéspedes en 4 segmentos con k-means (`grupo`, una etiqueta de 0 a 3 por huésped) para poder colorear después los gráficos.
- **Qué debe salir:**
    - `(1200, 7)` (1 200 huéspedes y 7 variables) y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno. Los 4 segmentos son los del cuaderno de clustering (S09_01).
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. La idea de PCA

#### A2 · Ejemplo — La varianza explicada de cada componente

**Analogía.** Imagina una nube de puntos con forma de puro. Para describirla no necesitas los tres ejes de la habitación: te basta **un eje a lo largo del puro** (donde hay más variación) y otro más corto a lo ancho. PCA encuentra esos nuevos ejes:
- el **componente 1** es la dirección con **más variación** de los datos,
- el **componente 2** es la siguiente, **perpendicular** a la primera, y así sucesivamente.

Cada componente es una **mezcla** de las variables originales, y viene con un porcentaje de **varianza explicada** (cuánta información conserva).

- **Qué hace:**
    - `PCA(random_state=SEMILLA).fit(X)` calcula tantos componentes como variables (7).
    - `explained_variance_ratio_` da la proporción de varianza que explica cada componente (`varianza`); el bucle las escribe en porcentaje.
- **Qué debe salir:**
    - Siete líneas: `Componente 1: 43.2%` · `Componente 2: 25.5%` · `Componente 3: 12.6%` · `Componente 4: 8.5%` · `Componente 5: 4.1%` · `Componente 6: 3.4%` · `Componente 7: 2.8%`.
- **Cómo leerlo:**
    - Los porcentajes bajan de mayor a menor y suman 100 %. El primer componente por sí solo ya explica el 43 % de la información de las 7 variables.
- **Si falla:**
    - `NameError: name 'X' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué porcentaje de la información explica el componente 1 y cuál el 2?* → **43,2 %** y **25,5 %**.
2. *¿Qué componentes explican menos del 5 % cada uno?* → Del 5 al 7: **4,1 %, 3,4 % y 2,8 %**.
3. *¿Por qué los porcentajes están ordenados de mayor a menor?* → Porque PCA busca primero la dirección con **más variación** y luego, perpendicular a ella, la siguiente con más variación que quede, y así sucesivamente.

</details>

#### A3 · Ejercicio 1 — La varianza acumulada y el 90 %

- **Qué hace:**
    - `np.cumsum(varianza)` calcula la varianza **acumulada**: el primer valor es el del componente 1, el segundo suma los dos primeros, etc. (`varianza_acum`, 7 valores que acaban en 1).
    - `varianza_acum >= 0.90` da una lista de verdadero/falso; `np.argmax(...)` devuelve la **posición de la primera** casilla verdadera (empezando en 0).
    - `+ 1` convierte esa posición en un número de componentes (se cuenta desde 1), y `int(...)` lo pasa a número normal (`n_comp_90`).
- **Qué debe salir:**
    - `con 5 componentes conservamos el 90 % de la información` y `✔ Ejercicio 1 correcto`. Las acumuladas son 43,2 %, 68,7 %, 81,3 %, 89,8 %, 93,8 %, 97,2 % y 100 %.
- **Si falla:**
    - `varianza_acum debe acabar en 1 y tener 7 valores` → no has usado `np.cumsum(varianza)`.
    - `Revisa: busca el primer componente donde la acumulada llega a 0.90 y cuéntalo desde 1` → te has quedado en 4 (la posición, que empieza en 0) o has contado mal: con 4 componentes se llega al 89,8 %, justo por debajo del 90 %.
    - `NameError: name 'varianza' is not defined` → falta ejecutar el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valen `n_comp_90` y la varianza acumulada con 4 componentes?* → `n_comp_90 = 5`; con 4 componentes la acumulada es **89,8 %**.
2. *¿Por qué con 4 componentes no basta, si 89,8 % casi es 90 %?* → Porque el criterio es llegar **al menos** al 90 %: con 4 componentes se queda en 89,8 %, por debajo, y hace falta el quinto (93,8 %).
3. *¿Qué ahorro supone quedarse con 5 componentes?* → Pasar de **7 variables a 5** (un 29 % menos) conservando el **90 %** de la información.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
varianza_acum = np.cumsum(varianza)
n_comp_90 = int(np.argmax(varianza_acum >= 0.90)) + 1
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A4 · Ejemplo — Gráficos de la varianza: por componente y acumulada

- **Qué hace:**
    - A la izquierda, `ax[0].bar(...)` dibuja una barra por componente con su varianza explicada.
    - A la derecha, `ax[1].plot(...)` dibuja la varianza acumulada (`varianza_acum`) y `ax[1].axhline(0.9, ...)` traza la línea roja del 90 %.
- **Qué debe salir:**
    - Dos gráficos. Izquierda: siete barras que bajan desde 0,43 (componente 1) hasta 0,03 (componente 7). Derecha: una curva que sube de 0,43 a 1 y cruza la línea roja del 90 % en el 5.º punto.
- **Cómo leerlo:**
    - La curva de la derecha es la que sirve para decidir: se mira dónde corta la línea del 90 %. El «codo» (donde la curva deja de subir deprisa) está hacia el 3.º–4.º componente.
- **Si falla:**
    - `NameError: name 'varianza_acum' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿En qué componente cruza la curva la línea roja del 90 %?* → En el **5.º** (con 4 queda por debajo, 89,8 %).
2. *¿Qué ve el gráfico de barras que no ve el de la curva?* → Cuánto aporta **cada componente por separado**: las barras bajan rápido (43, 25, 13, 8, 4, 3, 3 %).
3. *¿Qué quiere decir que la curva «se aplane» al final?* → Que los últimos componentes **añaden muy poca información**: pasar de 5 a 7 componentes solo sube del 93,8 % al 100 %.

</details>

## 2. Ver a los huéspedes en 2D

#### A5 · Ejemplo — Proyectar los huéspedes a 2 componentes

- **Qué hace:**
    - `PCA(n_components=2, ...)` se queda solo con los 2 primeros componentes.
    - `fit_transform(X)` calcula los componentes y **proyecta** cada huésped sobre ellos: `Z2` tiene, para cada huésped, sus dos coordenadas (componente 1 y componente 2).
    - El `print` escribe la forma de `Z2` y la información que conservan los dos componentes (la suma de sus varianzas explicadas).
- **Qué debe salir:**
    - `Forma: (1200, 2) · información conservada: 68.7%`.
- **Cómo leerlo:**
    - `(1200, 2)`: 1 200 huéspedes con **2 coordenadas** en lugar de 7 variables. Con ellas sí se puede dibujar. El 68,7 % es la suma de 43,2 % y 25,5 %.
- **Si falla:**
    - `NameError: name 'X' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tiene `Z2` y qué significa cada dimensión?* → **(1200, 2)**: una fila por huésped y dos columnas, sus coordenadas en el componente 1 y el componente 2.
2. *¿Cuánta información se conserva con 2 componentes?* → **68,7 %** (43,2 % + 25,5 %).
3. *¿Qué se pierde al quedarse con 2 componentes?* → El **31,3 %** restante de la variación: lo que ves en el plano es una **sombra** de los datos.

</details>

#### A6 · Ejercicio 2 — El mapa de huéspedes en 2D

- **Qué hace:**
    - `plt.subplots(figsize=(7, 5))` crea la figura (`fig2d`) y sus ejes (`ax`).
    - `ax.scatter(Z2[:, 0], Z2[:, 1], c=grupo, cmap="tab10", ...)` dibuja un punto por huésped (columna 0 de `Z2` en el eje X, columna 1 en el Y), coloreado según su segmento.
    - `ax.set_xlabel(...)` y `ax.set_ylabel(...)` ponen el nombre de cada eje con su porcentaje de varianza (`pca2.explained_variance_ratio_`); `ax.set_title(...)` el título; `ax.legend(...)` la leyenda de segmentos.
- **Qué debe salir:**
    - Un gráfico de dispersión con 4 nubes de puntos bien separadas: el segmento 0 arriba a la izquierda, el 2 arriba a la derecha, el 3 abajo a la izquierda y el 1 abajo a la derecha. Ejes: «Componente 1 (43%)» y «Componente 2 (26%)». Después, `✔ Ejercicio 2 correcto`.
    - **Lectura:** los segmentos se ven **separados** en el plano; PCA permite *comprobar a ojo* que el clustering tiene sentido. Ojo: 2 componentes conservan un ~69 % de la información; lo que ves es una **sombra** de los datos, no toda la verdad.
- **Si falla:**
    - `fig2d debe contener un scatter` → no has dibujado nada en `fig2d` con `ax.scatter(...)` o has llamado a otra figura.
    - `Añade un título al gráfico` → falta `ax.set_title(...)` (el título no puede estar vacío).
    - `NameError: name 'Z2' is not defined` → falta ejecutar el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Se ven los 4 segmentos separados? ¿Cuáles son los más cercanos entre sí?* → **Sí**, 4 nubes distintas. Los más cercanos son el **0** (arriba a la izquierda) y el **3** (abajo a la izquierda), que casi se tocan.
2. *¿Qué indica sobre el clustering que los segmentos salgan separados en el mapa?* → Que el clustering **tiene sentido**: los huéspedes de un mismo segmento están juntos también en el plano de PCA.
3. *¿Por qué conviene poner el % de varianza en los ejes?* → Porque dice **cuánta información** lleva cada eje: el componente 1 (≈ 43 %) lleva casi el doble que el 2 (≈ 26 %), y entre los dos solo se ve el 69 %.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
fig2d, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(Z2[:, 0], Z2[:, 1], c=grupo, cmap="tab10", s=12, alpha=0.7)
ax.set_xlabel(f"Componente 1 ({pca2.explained_variance_ratio_[0]:.0%})")
ax.set_ylabel(f"Componente 2 ({pca2.explained_variance_ratio_[1]:.0%})")
ax.set_title("Mapa de huéspedes (PCA) coloreado por segmento")
ax.legend(*sc.legend_elements(), title="Segmento")
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>